Let's say we have a function that inserts patient data into a database

In [1]:
def insert_patient_data(name, age):
    print(name)
    print(age)
    print("Data inserted into DB")

In [2]:
insert_patient_data("John Doe", 30)

John Doe
30
Data inserted into DB


In [3]:
insert_patient_data("John Doe", "30")

John Doe
30
Data inserted into DB


In [4]:
insert_patient_data("John Doe", "thirty")

John Doe
thirty
Data inserted into DB


This will not raise an error, but it will cause problems when we try to use the age as a number

For example, if we want to calculate the patient's birth year, we might do something like:

birth_year = 2023 - age

This will raise a TypeError because we cannot subtract a string from an integer

To avoid this problem, we need to validate the data before inserting it into the DB

In [5]:
def insert_patient_data(name, age):
    if not isinstance(name, str):
        raise ValueError("Name must be a string")
    if not isinstance(age, int):
        raise ValueError("Age must be an integer")
    print(name)
    print(age)
    print("Data inserted into DB")

In [6]:
insert_patient_data("John Doe", "30")

ValueError: Age must be an integer

In [7]:
insert_patient_data("John Doe", 30)

John Doe
30
Data inserted into DB


Also the age must always be a positive integer, so we can add another validation check for that

In [8]:
def insert_patient_data(name, age):
    if not isinstance(name, str):
        raise ValueError("Name must be a string")
    if not isinstance(age, int):
        raise ValueError("Age must be an integer")
    if age < 0:
        raise ValueError("Age must be a positive integer")
    print(name)
    print(age)
    print("Data inserted into DB")

In [9]:
insert_patient_data("John Doe", -30)

ValueError: Age must be a positive integer

## Pydantic

In [3]:
from pydantic import BaseModel

In [11]:
class Patient(BaseModel):
    name: str
    age: int

In [13]:
patient_info = {'name': "John Doe", 'age': 30}

In [15]:
patient1 = Patient(**patient_info)
print(patient1)

name='John Doe' age=30


In [16]:
def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print("Data inserted into DB")

In [17]:
insert_patient_data(patient1)

John Doe
30
Data inserted into DB


In [18]:
insert_patient_data(Patient(name="John Doe", age=30))

John Doe
30
Data inserted into DB


In [19]:
def update_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print("Data updated in DB")

In [20]:
update_patient_data(patient1)

John Doe
30
Data updated in DB


In [21]:
class Patient(BaseModel):
    name: str
    age: int
    weight: float

In [22]:
def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print("Data inserted into DB")

In [23]:
insert_patient_data(Patient(name="John Doe", age=30, weight=70.5))

John Doe
30
70.5
Data inserted into DB


What if I don't include weight?

In [24]:
insert_patient_data(Patient(name="John Doe", age=30))

ValidationError: 1 validation error for Patient
weight
  Field required [type=missing, input_value={'name': 'John Doe', 'age': 30}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

Pydantic is smart enought to convert the string to int for age and to float for weight


In [25]:
insert_patient_data(Patient(name="John Doe", age="30", weight="70.5"))

John Doe
30
70.5
Data inserted into DB


In [4]:
from typing import List, Dict

In [27]:
class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_info: Dict[str, str]

Why not used list for allergies & dict for contact_info?

Why used List[str] and Dict[str, str]??

list will only check if allergies is a list or not, while List[str] will check if all the elements inside the list are of type string or not? Similarly dict only checks if contact_info is a dictionary or not, while Dict[str, str] will check if all the keys and values inside the dictionary are of type string or not?

In [28]:
patient_info = {'name': 'benky', 'age': 30, 'weight': 70.5, 'married': False, 'allergies': ['pollen', 'dust'], 'contact_info': {'email': 'benky@example.com', 'phone': '123-456-7890'}}
patient2 = Patient(**patient_info)
print(patient2)

name='benky' age=30 weight=70.5 married=False allergies=['pollen', 'dust'] contact_info={'email': 'benky@example.com', 'phone': '123-456-7890'}


In [29]:
insert_patient_data(patient2)

benky
30
70.5
Data inserted into DB


What if I try to send one of the allergies as an integer?

In [31]:
patient_info = {'name': 'benky', 'age': '30', 'weight': '70.5', 'married': False, 'allergies': ['pollen', 121], 'contact_info': {'email': 'benky@example.com', 'phone': '123-456-7890'}}
patient3 = Patient(**patient_info)

ValidationError: 1 validation error for Patient
allergies.1
  Input should be a valid string [type=string_type, input_value=121, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

What if I send one of the values of the contact_info as an integer?

In [33]:
patient_info = {'name': 'benky', 'age': '30', 'weight': '70.5', 'married': False, 'allergies': ['pollen', 'dust'], 'contact_info': {'email': 'benky@example.com', 'phone': 1234567890}}
patient4 = Patient(**patient_info)

ValidationError: 1 validation error for Patient
contact_info.phone
  Input should be a valid string [type=string_type, input_value=1234567890, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

Required & Optional fields

In [5]:
from typing import Optional

In [41]:
class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: Optional[bool] = False
    allergies: Optional[List[str]] = None
    contact_info: Dict[str, str]

Removed allergies

In [36]:
patient_info = {'name': 'benky', 'age': '30', 'weight': '70.5', 'married': False, 'contact_info': {'email': 'benky@example.com', 'phone': '1234567890'}}
patient5 = Patient(**patient_info)

In [6]:
def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    if patient.allergies:
        print(patient.allergies)
    print(patient.contact_info)
    print("Data inserted into DB")

NameError: name 'Patient' is not defined

In [38]:
insert_patient_data(patient5)

benky
30
70.5
False
{'email': 'benky@example.com', 'phone': '1234567890'}
Data inserted into DB


In [42]:
patient_info = {'name': 'benky', 'age': '30', 'weight': '70.5', 'contact_info': {'email': 'benky@example.com', 'phone': '1234567890'}}
patient6 = Patient(**patient_info)

In [43]:
insert_patient_data(patient6)

benky
30
70.5
False
{'email': 'benky@example.com', 'phone': '1234567890'}
Data inserted into DB


Data Validation

In [1]:
from pydantic import BaseModel, EmailStr
from typing import List, Dict, Optional

In [4]:
%pip install "pydantic[email]"

  Using cached email_validator-2.3.0-py3-none-any.whl.metadata (26 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
Using cached email_validator-2.3.0-py3-none-any.whl (35 kB)
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python311.exe -m pip install --upgrade pip


In [5]:
class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: Optional[bool] = False
    allergies: Optional[List[str]] = None
    contact_info: Dict[str, str]

In [6]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'age': '30', 'weight': '70.5', 'contact_info': {'phone': '1234567890'}}
patient7 = Patient(**patient_info)

What if I give an invalid email id?

In [10]:
patient_info = {'name': 'benky', 'email': 'benkyexample.com', 'age': '30', 'weight': '70.5', 'contact_info': {'phone': '1234567890'}}
patient8 = Patient(**patient_info)

ValidationError: 2 validation errors for Patient
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='benkyexample.com', input_type=str]
linkedin_url
  Field required [type=missing, input_value={'name': 'benky', 'email'...{'phone': '1234567890'}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [8]:
from pydantic import AnyUrl

In [9]:
class Patient(BaseModel):
    name: str
    email: EmailStr
    linkedin_url: AnyUrl
    age: int
    weight: float
    married: Optional[bool] = False
    allergies: Optional[List[str]] = None
    contact_info: Dict[str, str]

In [11]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'contact_info': {'phone': '1234567890'}}
patient9 = Patient(**patient_info)

What if I provide an invalid URL?

In [13]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'contact_info': {'phone': '1234567890'}}
patient10 = Patient(**patient_info)

ValidationError: 1 validation error for Patient
linkedin_url
  Input should be a valid URL, relative URL without a base [type=url_parsing, input_value='linkedin.com/in/benky', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/url_parsing

Constraints using Field

In [14]:
from pydantic import Field

In [23]:
class Patient(BaseModel):
    name: str = Field(max_length=50)
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: float = Field(gt=0)
    married: Optional[bool] = False
    allergies: Optional[List[str]] = Field(max_length=2)
    contact_info: Dict[str, str]

Defined allergies elements max length as 2 elements, thus pollen must raise a Validation Error

In [24]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'allergies': ['dust', 'pollen', 'peanuts'], 'contact_info': {'phone': '1234567890'}}
patient12 = Patient(**patient_info)

ValidationError: 1 validation error for Patient
allergies
  List should have at most 2 items after validation, not 3 [type=too_long, input_value=['dust', 'pollen', 'peanuts'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/too_long

Meta Data using Field

In [25]:
from typing import Annotated

Added title, description and examples

In [27]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: float = Field(gt=0)
    married: Optional[bool] = False
    allergies: Optional[List[str]] = Field(max_length=2)
    contact_info: Dict[str, str]

In [29]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient13 = Patient(**patient_info)

In [30]:
print(patient13)

name='benky' email='benky@example.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


We can also set default values using Field

In [28]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: float = Field(gt=0)
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Optional[List[str]] = Field(max_length=2)
    contact_info: Dict[str, str]

In [31]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient14 = Patient(**patient_info)

In [32]:
print(patient14)

name='benky' email='benky@example.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


Type Coercion

In [33]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: float = Field(gt=0)
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

Even if I send weight as string, pydantic is accepting and converting it to the float

In [34]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient15 = Patient(**patient_info)
print(patient15)

name='benky' email='benky@example.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=30 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890'}


What if you want to suppress the default type coercion?

In [36]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

In [37]:
patient_info = {'name': 'benky', 'email': 'benky@example.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '30', 'weight': '70.5', 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890'}}
patient16 = Patient(**patient_info)
print(patient16)

ValidationError: 1 validation error for Patient
weight
  Input should be a valid number [type=float_type, input_value='70.5', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/float_type